# HoTHP vs RoTHP: Three Targeted Scientific Tests

Previous experiments tested temporal extrapolation on a standard Hawkes process and found
no consistent NLL advantage. This revealed two things:

1. For processes with **fast-decaying kernels** (beta_slow=0.15), both models learn to ignore
   distant events via compensatory value-vector learning. The inductive bias of HoTHP
   is not needed.
2. Per-sequence normalization makes both models **scale-invariant at the pipeline level**,
   hiding HoTHP's internal architectural advantage.

This notebook tests three targeted hypotheses designed to isolate scenarios where
HoTHP's design genuinely matters.

---

## Pre-registered Hypotheses

### H1 — Scale Invariance (guaranteed by architecture)
HoTHP's `_normalize_timestamps()` gives it **exact scale invariance**: regardless of how
timestamps are scaled, it always sees mean gap = 1.0. RoTHP has no such protection.

**Test**: Train with global normalization. Test at scale multipliers {0.1×, 1×, 5×, 10×, 50×}.
**Expected**: HoTHP NLL flat across scales. RoTHP degrades when scale ≠ 1.

### H2 — Sample Efficiency
HoTHP's inductive bias (monotonic decay) encodes the prior that recent events matter more.
RoTHP must discover this from data (compensatory value-vector learning).

**Test**: Train on N ∈ {25, 50, 100, 250, 500} sequences. Measure test NLL vs N.
**Expected**: HoTHP reaches good NLL at smaller N, especially on long-range processes.

### H3 — Long-Range Process
When the true Hawkes kernel has **slow decay** (influence persists for 100+ normalized
lags), HoTHP's monotonic basis represents it directly. RoTHP's oscillatory basis must
approximate it through interference — harder with limited model capacity.

**Test**: Compare on slow-decay process (β_normalized ≈ 0.01) vs fast-decay baseline.
**Expected**: HoTHP NLL advantage is larger on slow-decay, especially at small N.

In [ ]:
import os, sys, math, random, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import contextlib

BASE_SEED = 42
N_SEEDS   = 5

def set_global_seed(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)

def make_run_seed(*parts, base_seed=BASE_SEED):
    key = '::'.join(map(str, parts))
    return (base_seed + int(hashlib.sha256(key.encode()).hexdigest()[:8], 16)) % (2**31)

set_global_seed(BASE_SEED)
sns.set_theme(style='whitegrid')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if not os.path.exists('ufc-easytpp'):
    !git clone https://github.com/hugoramos/ufc-easytpp.git
if 'ufc-easytpp' not in sys.path:
    sys.path.insert(0, os.path.abspath('ufc-easytpp'))

import easy_tpp.model.torch_model.torch_baselayer as baselayer

def attention_fixed(query, key, value, mask=None, dropout=None):
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / (d_k ** 0.5)
    if mask is not None:
        if mask.dim() == 3: mask = mask.unsqueeze(1)
        scores = scores.masked_fill(mask > 0, -1e4)
    p_attn = torch.softmax(scores, dim=-1)
    if dropout is not None: p_attn = dropout(p_attn)
    return torch.matmul(p_attn, value), p_attn

baselayer.attention = attention_fixed
import easy_tpp.model.torch_model.torch_rothp as rothp_module
rothp_module.attention = attention_fixed

from easy_tpp.config_factory.model_config import ModelConfig
from easy_tpp.model.torch_model.torch_rothp import RoTHP
from easy_tpp.model.torch_model.torch_hothp import HoTHP

NUM_TYPES = 2  # simplified for clarity
pad_id    = NUM_TYPES

config = ModelConfig(**{
    'hidden_size': 32, 'num_layers': 2, 'num_heads': 2, 'dropout_rate': 0.1,
    'num_event_types': NUM_TYPES, 'num_event_types_pad': NUM_TYPES + 1,
    'event_pad_index': pad_id, 'time_emb_size': 32, 'use_ln': True,
    'gpu': 0 if torch.cuda.is_available() else -1, 'model_id': 'ScientificCase',
    'thinning': {'num_sample':1,'num_exp':500,'over_sample_rate':5.0,
                 'patience_counter':5,'num_samples_boundary':5,'dtime_max':5.0,'num_step_gen':1},
    'loss_integral_num_sample_per_step': 20, 'use_mc_samples': False,
})

USE_AMP = device.type == 'cuda'
if USE_AMP:
    try:
        _scaler_cls = torch.amp.GradScaler
        _autocast_fn = lambda: torch.amp.autocast(device_type='cuda')
    except AttributeError:
        _scaler_cls = torch.cuda.amp.GradScaler
        _autocast_fn = lambda: torch.cuda.amp.autocast()
else:
    _scaler_cls = None
    _autocast_fn = contextlib.nullcontext

print(f'Device: {device}  |  AMP: {USE_AMP}')

In [ ]:
# ── Process definitions ──────────────────────────────────────────────────────
#
# FAST-DECAY process (baseline)
#   beta_raw=0.5, mean_gap_raw≈0.5  →  beta_normalized ≈ 0.25
#   Influence at normalized lag 20: exp(-0.25×20) = exp(-5) ≈ 0.007  (negligible)
#
# SLOW-DECAY process (long-range)
#   beta_raw=0.02, mean_gap_raw≈0.5  →  beta_normalized ≈ 0.01
#   Influence at normalized lag 20:  exp(-0.01×20) = 0.82  (still strong)
#   Influence at normalized lag 100: exp(-0.01×100) = 0.37  (substantial)
#   Influence at normalized lag 200: exp(-0.01×200) = 0.14  (still detectable)

PROC_FAST = dict(
    mu     = np.array([0.4, 0.4]),
    alpha  = np.array([[0.12, 0.08], [0.08, 0.12]]),
    beta   = 0.5,
    label  = 'fast-decay (β_norm≈0.25)',
)

PROC_SLOW = dict(
    mu     = np.array([0.3, 0.3]),
    alpha  = np.array([[0.008, 0.006], [0.006, 0.008]]),  # must be small for stability
    beta   = 0.02,
    label  = 'slow-decay (β_norm≈0.01)',
)


def generate_hawkes(rng, proc, horizon, min_events=20, max_events=200):
    mu, alpha, beta = proc['mu'], proc['alpha'], proc['beta']
    n_types = len(mu)
    while True:
        events, t = [], 0.0
        while t < horizon and len(events) < max_events:
            intensity = mu.copy()
            for t_i, k_i in events:
                intensity += alpha[:, k_i] * np.exp(-beta * (t - t_i))
            lam_bar = float(np.sum(intensity))
            if lam_bar <= 1e-9: break
            t += rng.exponential(1.0 / lam_bar)
            if t >= horizon: break
            candidate = mu.copy()
            for t_i, k_i in events:
                candidate += alpha[:, k_i] * np.exp(-beta * (t - t_i))
            lam_sum = float(np.sum(candidate))
            if rng.uniform() <= lam_sum / lam_bar:
                probs = candidate / lam_sum
                events.append((t, int(rng.choice(n_types, p=probs))))
        if len(events) >= min_events:
            return events[:max_events]


def make_dataset(rng, proc, n_seqs, horizon, min_events, max_events):
    return [generate_hawkes(rng, proc, horizon, min_events, max_events)
            for _ in range(n_seqs)]


# Verify beta_normalized for each process
rng0 = np.random.default_rng(BASE_SEED)
for proc in [PROC_FAST, PROC_SLOW]:
    seqs = make_dataset(rng0, proc, 200, horizon=50.0, min_events=20, max_events=200)
    all_gaps = []
    for seq in seqs:
        times = sorted([t for t, _ in seq])
        all_gaps.extend([times[i]-times[i-1] for i in range(1, len(times))])
    mean_gap = np.mean(all_gaps)
    beta_norm = proc['beta'] * mean_gap
    inf_at_100 = math.exp(-beta_norm * 100)
    print(f"{proc['label']}")
    print(f"  mean_gap_raw={mean_gap:.3f}, beta_normalized={beta_norm:.4f}")
    print(f"  Influence at normalized lag 100: {inf_at_100:.3f}")
    print(f"  Influence at normalized lag  20: {math.exp(-beta_norm*20):.3f}")

In [ ]:
# ── Shared utilities ─────────────────────────────────────────────────────────

def to_tensors_global_norm(seqs, global_mean_gap):
    """Normalize using a GLOBAL mean gap (from training data).
    At test time with a different scale, this does NOT re-normalize —
    exposing RoTHP to scale shift while HoTHP compensates internally.
    """
    converted = []
    for seq in seqs:
        seq = sorted(seq, key=lambda x: x[0])
        times  = torch.tensor([t for t, _ in seq], dtype=torch.float32)
        types  = torch.tensor([k for _, k in seq], dtype=torch.long)
        deltas = torch.zeros_like(times)
        deltas[1:] = times[1:] - times[:-1]
        times  = (times - times[0]) / global_mean_gap
        deltas = deltas / global_mean_gap
        converted.append({'time_seqs': times, 'time_delta_seqs': deltas, 'type_seqs': types})
    return converted


def to_tensors_per_seq(seqs):
    """Per-sequence normalization (mean gap = 1.0). Used for H2 and H3."""
    converted = []
    for seq in seqs:
        seq = sorted(seq, key=lambda x: x[0])
        times  = torch.tensor([t for t, _ in seq], dtype=torch.float32)
        types  = torch.tensor([k for _, k in seq], dtype=torch.long)
        deltas = torch.zeros_like(times)
        deltas[1:] = times[1:] - times[:-1]
        mean_gap = deltas[1:].mean().clamp(min=1e-6)
        times  = (times - times[0]) / mean_gap
        deltas = deltas / mean_gap
        converted.append({'time_seqs': times, 'time_delta_seqs': deltas, 'type_seqs': types})
    return converted


def collate_fn(batch_list):
    B = len(batch_list)
    L = max(len(x['time_seqs']) for x in batch_list)
    pad_time  = torch.zeros(B, L, dtype=torch.float32)
    pad_delta = torch.zeros(B, L, dtype=torch.float32)
    pad_type  = torch.full((B, L), pad_id, dtype=torch.long)
    npm       = torch.zeros(B, L, dtype=torch.float32)
    attn      = torch.ones(B, L, L, dtype=torch.bool)
    causal    = torch.triu(torch.ones(L, L, dtype=torch.bool), diagonal=1)
    for i, item in enumerate(batch_list):
        l = len(item['time_seqs'])
        pad_time[i, :l]  = item['time_seqs']
        pad_delta[i, :l] = item['time_delta_seqs']
        pad_type[i, :l]  = item['type_seqs']
        npm[i, :l] = 1.0
        m = causal.clone(); m[:, l:] = True; m[l:, :] = True
        attn[i] = m
    return (pad_time, pad_delta, pad_type, npm, attn)


def make_loader(data, batch_size, shuffle=False, seed=None):
    gen = None
    if shuffle and seed is not None:
        gen = torch.Generator(); gen.manual_seed(seed)
    return DataLoader(data, batch_size=batch_size, shuffle=shuffle,
                      collate_fn=collate_fn, generator=gen)


def evaluate_nll(model, loader):
    model.eval()
    total_loss, total_events = 0.0, 0
    with torch.no_grad():
        for batch in loader:
            batch = [t.to(device) for t in batch]
            with _autocast_fn():
                loss, num = model.loglike_loss(batch)
            total_loss += loss.item()
            total_events += num
    return total_loss / (total_events + 1e-9)


def train_model(model, train_loader, val_loader, epochs=400, patience=20,
                grad_clip=1.0, lr=1e-3):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode='min', factor=0.5, patience=8, min_lr=1e-5)
    scaler = _scaler_cls(enabled=True) if USE_AMP else None
    best_val, best_state, no_improve = float('inf'), None, 0
    history = []

    for ep in range(epochs):
        model.train()
        for batch in train_loader:
            batch = [t.to(device) for t in batch]
            opt.zero_grad()
            with _autocast_fn():
                loss, num = model.loglike_loss(batch)
                nll = loss / (num + 1e-9)
            if not torch.isnan(nll):
                if scaler:
                    scaler.scale(nll).backward()
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                    scaler.step(opt); scaler.update()
                else:
                    nll.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                    opt.step()

        val_nll = evaluate_nll(model, val_loader)
        sched.step(val_nll)
        history.append({'ep': ep+1, 'val_nll': val_nll,
                        'lr': opt.param_groups[0]['lr']})

        if val_nll < best_val - 1e-4:
            best_val = val_nll
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
        if no_improve >= patience:
            break

    if best_state:
        model.load_state_dict(best_state)
    return best_val, history


print('Utilities ready.')

---
## Experiment H1: Scale Invariance

**Setup**: Train with global normalization (training mean gap).  
At test time, multiply all raw timestamps by a scale factor **s** before applying  
the *same* global normalization constant from training.  
This means the test data has mean gap = s (instead of 1.0 during training).

- **RoTHP**: receives gaps of size s → cos(s × gap × θⱼ) has wrong phase → degrades
- **HoTHP**: `_normalize_timestamps()` divides by per-sequence mean gap = s → always sees 1.0 → **invariant**

This is not just expected — it is **provably true** by inspection of the code.

In [ ]:
TEST_SCALES = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 50.0]
seeds = [BASE_SEED + i * 100 for i in range(N_SEEDS)]

h1_results = []  # list of {seed, scale, rothp_nll, hothp_nll}

rng_h1 = np.random.default_rng(make_run_seed('H1', 'data'))

# Generate data once
train_raw = make_dataset(rng_h1, PROC_FAST, 500, horizon=20.0, min_events=20, max_events=100)
val_raw   = make_dataset(rng_h1, PROC_FAST, 100, horizon=20.0, min_events=20, max_events=100)
test_raw  = make_dataset(rng_h1, PROC_FAST, 200, horizon=20.0, min_events=20, max_events=100)

# Compute global mean gap from training data
all_gaps = []
for seq in train_raw:
    times = sorted([t for t, _ in seq])
    all_gaps.extend([times[i]-times[i-1] for i in range(1, len(times))])
global_mean_gap = float(np.mean(all_gaps))
print(f'Global mean gap (training): {global_mean_gap:.4f}')

# Pre-process training and validation (global norm, scale=1)
train_data = to_tensors_global_norm(train_raw, global_mean_gap)
val_data   = to_tensors_global_norm(val_raw,   global_mean_gap)

for seed_idx, seed in enumerate(seeds):
    print(f'\n--- Seed {seed_idx+1}/{N_SEEDS} (seed={seed}) ---')

    train_loader = make_loader(train_data, 64, shuffle=True,
                               seed=make_run_seed('H1', seed, 'loader'))
    val_loader   = make_loader(val_data,   64)

    # Train RoTHP
    set_global_seed(make_run_seed('H1', 'RoTHP', base_seed=seed))
    rothp = RoTHP(config).to(device)
    train_model(rothp, train_loader, val_loader, epochs=300, patience=20, lr=1e-3)

    # Train HoTHP
    set_global_seed(make_run_seed('H1', 'HoTHP', base_seed=seed))
    hothp = HoTHP(config).to(device)
    train_model(hothp, train_loader, val_loader, epochs=300, patience=20, lr=5e-4)

    # Evaluate at each test scale
    for s in TEST_SCALES:
        # Scale the raw test timestamps by s, then apply the same global_mean_gap normalization
        test_scaled = []
        for seq in test_raw:
            seq_scaled = [(t * s, k) for t, k in seq]
            test_scaled.append(seq_scaled)
        test_data_s = to_tensors_global_norm(test_scaled, global_mean_gap)
        test_loader = make_loader(test_data_s, 64)

        r_nll = evaluate_nll(rothp, test_loader)
        h_nll = evaluate_nll(hothp, test_loader)

        h1_results.append({'seed': seed, 'scale': s,
                           'rothp_nll': r_nll, 'hothp_nll': h_nll})
        print(f'  scale={s:6.1f}x: RoTHP={r_nll:.4f}  HoTHP={h_nll:.4f}')

print('\nH1 complete.')

In [ ]:
h1_df = pd.DataFrame(h1_results)

# Normalise by in-distribution (scale=1) NLL to show *relative* degradation
ref = h1_df[h1_df['scale'] == 1.0].set_index('seed')[['rothp_nll', 'hothp_nll']]
rows = []
for _, row in h1_df.iterrows():
    r_ref = ref.loc[row['seed'], 'rothp_nll']
    h_ref = ref.loc[row['seed'], 'hothp_nll']
    rows.append({'seed': row['seed'], 'scale': row['scale'],
                 'rothp_delta': row['rothp_nll'] - r_ref,
                 'hothp_delta': row['hothp_nll'] - h_ref})
delta_df = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, label, color in [
    (axes[0], 'rothp_nll', 'RoTHP', '#4C72B0'),
    (axes[0], 'hothp_nll', 'HoTHP', '#C44E52'),
]:
    grp = h1_df.groupby('scale')[col]
    means = grp.mean()
    stds  = grp.std()
    ax.plot(np.log10(TEST_SCALES), means.values, 'o-', color=color, lw=2, ms=7, label=label)
    ax.fill_between(np.log10(TEST_SCALES),
                    means.values - stds.values, means.values + stds.values,
                    color=color, alpha=0.12)

axes[0].axvline(0, color='gray', ls='--', alpha=0.6, label='training scale (1×)')
axes[0].set_xticks(np.log10(TEST_SCALES))
axes[0].set_xticklabels([f'{s}×' for s in TEST_SCALES])
axes[0].set_xlabel('Test time scale multiplier')
axes[0].set_ylabel('Test NLL')
axes[0].set_title('H1: Absolute NLL vs test scale')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Panel 2: NLL delta relative to scale=1
for col, label, color in [
    ('rothp_delta', 'RoTHP', '#4C72B0'),
    ('hothp_delta', 'HoTHP', '#C44E52'),
]:
    grp   = delta_df.groupby('scale')[col]
    means = grp.mean()
    stds  = grp.std()
    axes[1].plot(np.log10(TEST_SCALES), means.values, 'o-', color=color, lw=2, ms=7, label=label)
    axes[1].fill_between(np.log10(TEST_SCALES),
                         means.values - stds.values, means.values + stds.values,
                         color=color, alpha=0.12)

axes[1].axhline(0, color='gray', ls='--', alpha=0.6)
axes[1].axvline(0, color='gray', ls='--', alpha=0.4)
axes[1].set_xticks(np.log10(TEST_SCALES))
axes[1].set_xticklabels([f'{s}×' for s in TEST_SCALES])
axes[1].set_xlabel('Test time scale multiplier')
axes[1].set_ylabel('ΔNLL relative to 1× (lower is more robust)')
axes[1].set_title('H1: Degradation from scale shift\n(HoTHP should stay near 0)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('H1 — Scale Invariance: HoTHP is protected by _normalize_timestamps\n'
             'RoTHP sees wrong oscillation frequency → degrades', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('H1_Scale_Invariance.png', dpi=300, bbox_inches='tight')
plt.show()

print('H1 result summary (mean ΔNLL at each scale):')
print(delta_df.groupby('scale')[['rothp_delta','hothp_delta']].mean().to_string())

---
## Experiments H2 + H3: Sample Efficiency on Fast vs Slow Processes

**H2**: With fewer training samples, HoTHP's inductive bias (monotonic decay = recent events
matter more) should outperform RoTHP, which must discover this from data.

**H3**: This effect should be **larger for the slow-decay process**, where the model must
learn to represent long-range influence. HoTHP's monotonic basis directly represents
slow decay. RoTHP's oscillatory basis must superimpose 8 cosines to approximate it —
a harder optimization problem, especially at small N.

**Combined test**: Plot test NLL vs N_train for both models on both processes.
Expected pattern:
- Fast process: small or no HoTHP advantage (both models learn quickly)
- Slow process: clear HoTHP advantage at small N (inductive bias crucial)

In [ ]:
N_TRAIN_SIZES = [25, 50, 100, 250, 500]

rng_h23 = np.random.default_rng(make_run_seed('H2H3', 'data'))

# Generate all data upfront (use per-sequence normalization here)
datasets_h23 = {}
for proc_name, proc, horizon in [
    ('fast', PROC_FAST, 20.0),
    ('slow', PROC_SLOW, 80.0),  # longer horizon for slow process (more events per seq)
]:
    train_all = make_dataset(rng_h23, proc, max(N_TRAIN_SIZES), horizon,
                             min_events=20, max_events=150)
    val_raw   = make_dataset(rng_h23, proc, 150, horizon, min_events=20, max_events=150)
    test_raw  = make_dataset(rng_h23, proc, 200, horizon, min_events=20, max_events=150)

    datasets_h23[proc_name] = {
        'train_all': to_tensors_per_seq(train_all),
        'val':       to_tensors_per_seq(val_raw),
        'test':      to_tensors_per_seq(test_raw),
        'label':     proc['label'],
    }
    lens = [len(s) for s in train_all]
    print(f'{proc_name}: train len mean={np.mean(lens):.0f}, '
          f'min={min(lens)}, max={max(lens)}')


h23_results = []  # {proc, n_train, seed, rothp_nll, hothp_nll}

for proc_name in ['fast', 'slow']:
    d = datasets_h23[proc_name]
    val_loader  = make_loader(d['val'],  64)
    test_loader = make_loader(d['test'], 64)

    print(f'\n=== Process: {d["label"]} ===')

    for n_train in N_TRAIN_SIZES:
        train_subset = d['train_all'][:n_train]

        for seed in seeds:
            train_loader = make_loader(train_subset, min(32, n_train),
                                       shuffle=True,
                                       seed=make_run_seed('H23', proc_name, n_train, seed))

            set_global_seed(make_run_seed('H23', proc_name, 'RoTHP', n_train, base_seed=seed))
            rothp = RoTHP(config).to(device)
            train_model(rothp, train_loader, val_loader, epochs=400, patience=25, lr=1e-3)
            r_nll = evaluate_nll(rothp, test_loader)

            set_global_seed(make_run_seed('H23', proc_name, 'HoTHP', n_train, base_seed=seed))
            hothp = HoTHP(config).to(device)
            train_model(hothp, train_loader, val_loader, epochs=400, patience=25, lr=5e-4)
            h_nll = evaluate_nll(hothp, test_loader)

            h23_results.append({'proc': proc_name, 'n_train': n_train, 'seed': seed,
                                'rothp_nll': r_nll, 'hothp_nll': h_nll,
                                'advantage': r_nll - h_nll})  # positive = HoTHP wins

        grp = [r for r in h23_results if r['proc']==proc_name and r['n_train']==n_train]
        r_m = np.mean([r['rothp_nll'] for r in grp])
        h_m = np.mean([r['hothp_nll'] for r in grp])
        print(f'  N={n_train:4d}: RoTHP={r_m:.4f}  HoTHP={h_m:.4f}  '
              f'HoTHP advantage={r_m-h_m:+.4f}')

print('\nH2+H3 complete.')

In [ ]:
from scipy import stats

h23_df = pd.DataFrame(h23_results)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for row_idx, proc_name in enumerate(['fast', 'slow']):
    proc_df = h23_df[h23_df['proc'] == proc_name]
    label   = datasets_h23[proc_name]['label']

    # Panel A: NLL vs N_train
    ax = axes[row_idx, 0]
    for col, model_label, color in [
        ('rothp_nll', 'RoTHP', '#4C72B0'),
        ('hothp_nll', 'HoTHP', '#C44E52'),
    ]:
        grp   = proc_df.groupby('n_train')[col]
        means = grp.mean()
        stds  = grp.std()
        ax.plot(means.index, means.values, 'o-', color=color, lw=2, ms=7, label=model_label)
        ax.fill_between(means.index, means-stds, means+stds, color=color, alpha=0.12)
    ax.set_xscale('log')
    ax.set_xlabel('N training sequences (log scale)')
    ax.set_ylabel('Test NLL')
    ax.set_title(f'H2+H3 NLL vs N_train\n{label}')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Panel B: HoTHP advantage (RoTHP NLL - HoTHP NLL) vs N_train
    ax = axes[row_idx, 1]
    grp   = proc_df.groupby('n_train')['advantage']
    means = grp.mean()
    stds  = grp.std()
    ax.bar(range(len(means)), means.values,
           yerr=stds.values, color='#55A868', capsize=4, alpha=0.8)
    ax.axhline(0, color='gray', ls='--', alpha=0.7)
    ax.set_xticks(range(len(means)))
    ax.set_xticklabels([f'N={n}' for n in means.index])
    ax.set_ylabel('HoTHP advantage (RoTHP NLL − HoTHP NLL)')
    ax.set_title(f'H2+H3 HoTHP advantage\n{label}  (positive = HoTHP wins)')
    ax.grid(True, alpha=0.3, axis='y')

    # Significance at smallest N
    for n in N_TRAIN_SIZES:
        sub = proc_df[proc_df['n_train'] == n]
        if len(sub) >= 3:
            t, p2 = stats.ttest_rel(sub['rothp_nll'].values, sub['hothp_nll'].values)
            p1 = p2/2 if t > 0 else 1 - p2/2
            print(f'{proc_name} N={n}: advantage={sub["advantage"].mean():+.4f}, '
                  f'one-sided p={p1:.3f}')

plt.suptitle('H2+H3: Sample Efficiency — HoTHP advantage should be larger at small N\n'
             'and larger for slow-decay (long-range) process',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('H2H3_Sample_Efficiency.png', dpi=300, bbox_inches='tight')
plt.show()

---
## Attention Profile vs True Hawkes Kernel (H3 supporting evidence)

Even if NLL differences are small, HoTHP's learned attention should be
**structurally closer to the true Hawkes kernel** on the slow-decay process.

This is qualitative evidence that HoTHP's representations are more interpretable
and physically meaningful — a valid scientific contribution independent of NLL.

In [ ]:
# Train one pair of models (best seed) on each process and compare profiles
def extract_attention_vs_lag(model, loader, model_type, max_batches=30):
    model.eval()
    lags, weights = [], []
    with torch.no_grad():
        for b_idx, batch in enumerate(loader):
            if b_idx >= max_batches: break
            pad_time, pad_delta, pad_type, mask, attn_mask = [t.to(device) for t in batch]
            enc = model.layer_type_emb(pad_type)
            layer = model.stack_layers[0]
            if model_type == 'rothp':
                cos, sin = model.rotary_emb(pad_time)
                _, attn_w = layer.self_attn(enc, enc, enc, attn_mask,
                                            cos=cos, sin=sin, output_weight=True)
            else:
                nt = model._normalize_timestamps(pad_time)
                _, attn_w = layer.self_attn(enc, enc, enc, attn_mask,
                                            time_seqs=nt,
                                            thetas=model.hope_emb.thetas,
                                            theta_prime=model.hope_emb.theta_prime,
                                            output_weight=True)
            attn_w = attn_w.mean(dim=1).cpu()
            pt = pad_time.cpu(); m = mask.cpu()
            for b in range(pt.shape[0]):
                sl = int(m[b].sum().item())
                for i in range(sl):
                    for j in range(i):
                        lag = float(pt[b, i] - pt[b, j])
                        if lag > 0:
                            lags.append(lag)
                            weights.append(float(attn_w[b, i, j]))
    return np.array(lags), np.array(weights)


def bin_attn(lags, weights, n_bins=50):
    q98 = np.quantile(lags, 0.98)
    edges = np.linspace(0, q98, n_bins+1)
    centers = (edges[:-1]+edges[1:])/2
    means, stds = [], []
    for i in range(n_bins):
        m = (lags>=edges[i])&(lags<edges[i+1])
        means.append(weights[m].mean() if m.sum()>5 else np.nan)
        stds.append(weights[m].std() if m.sum()>5 else np.nan)
    return centers, np.array(means), np.array(stds)


fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for row_idx, (proc_name, proc_def) in enumerate([
    ('fast', PROC_FAST), ('slow', PROC_SLOW)
]):
    d = datasets_h23[proc_name]
    test_loader = make_loader(d['test'], 32)
    val_loader  = make_loader(d['val'],  64)

    # Train on full N=500 for best-quality profiles
    train_loader = make_loader(d['train_all'], 64, shuffle=True,
                               seed=make_run_seed('profile', proc_name))

    set_global_seed(make_run_seed('profile', proc_name, 'RoTHP'))
    rothp_p = RoTHP(config).to(device)
    train_model(rothp_p, train_loader, val_loader, epochs=400, patience=25, lr=1e-3)

    set_global_seed(make_run_seed('profile', proc_name, 'HoTHP'))
    hothp_p = HoTHP(config).to(device)
    train_model(hothp_p, train_loader, val_loader, epochs=400, patience=25, lr=5e-4)

    r_lags, r_w = extract_attention_vs_lag(rothp_p, test_loader, 'rothp')
    h_lags, h_w = extract_attention_vs_lag(hothp_p, test_loader, 'hothp')

    rc, rm, rs = bin_attn(r_lags, r_w)
    hc, hm, hs = bin_attn(h_lags, h_w)

    # True Hawkes kernel at same lag points
    beta_norm = proc_def['beta'] * np.mean([float(item['time_delta_seqs'][1:].mean())
                                            for item in d['test'][:50]])
    true_kernel = np.exp(-beta_norm * rc)
    true_kernel /= true_kernel.max()  # normalise for visual comparison

    # Panel: attention profiles
    ax = axes[row_idx, 0]
    ax.plot(rc, rm, color='#4C72B0', lw=1.8, label='RoTHP')
    ax.fill_between(rc, rm-rs, rm+rs, color='#4C72B0', alpha=0.12)
    ax.plot(hc, hm, color='#C44E52', lw=1.8, label='HoTHP')
    ax.fill_between(hc, hm-hs, hm+hs, color='#C44E52', alpha=0.12)
    ax.set_xlabel('Temporal lag (normalised)')
    ax.set_ylabel('Mean attention weight')
    ax.set_title(f'Learned attention profile\n{proc_def["label"]}')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Panel: vs true kernel
    ax = axes[row_idx, 1]
    valid = ~np.isnan(hm)
    hm_norm = hm[valid] / np.nanmax(hm) if np.nanmax(hm) > 0 else hm[valid]
    rm_norm = rm[valid] / np.nanmax(rm) if np.nanmax(rm) > 0 else rm[valid]
    ax.plot(rc[valid], true_kernel[valid], 'k--', lw=2, label='True Hawkes kernel (normalised)')
    ax.plot(rc[valid], rm_norm, color='#4C72B0', lw=1.8, label='RoTHP (normalised)')
    ax.plot(rc[valid], hm_norm, color='#C44E52', lw=1.8, label='HoTHP (normalised)')
    ax.set_xlabel('Temporal lag (normalised)')
    ax.set_ylabel('Normalised value')
    ax.set_title(f'Attention vs true kernel\n{proc_def["label"]}')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('H3: Does HoTHP\'s attention better match the true Hawkes kernel?\n'
             'Especially on slow-decay (long-range) process', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('H3_Attention_vs_Kernel.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
print('=' * 65)
print('SUMMARY: Three targeted tests for HoTHP vs RoTHP')
print('=' * 65)

# H1
h1_pivot = delta_df.groupby('scale')[['rothp_delta','hothp_delta']].mean()
max_r_delta = h1_pivot['rothp_delta'].abs().max()
max_h_delta = h1_pivot['hothp_delta'].abs().max()
print(f'\nH1 — Scale Invariance:')
print(f'  Max RoTHP degradation across scales: {max_r_delta:+.4f}')
print(f'  Max HoTHP degradation across scales: {max_h_delta:+.4f}')
conclusion = 'CONFIRMED' if max_r_delta > 3 * max_h_delta else 'PARTIAL/NOT CONFIRMED'
print(f'  → H1 {conclusion}')

# H2+H3
print(f'\nH2+H3 — Sample Efficiency:')
for proc_name in ['fast', 'slow']:
    sub = h23_df[h23_df['proc'] == proc_name]
    by_n = sub.groupby('n_train')['advantage'].mean()
    print(f'  {proc_name.upper()} process — HoTHP advantage by N_train:')
    for n, adv in by_n.items():
        bar = '▓' * max(0, int(adv * 200)) if adv > 0 else '░' * max(0, int(-adv * 200))
        print(f'    N={n:4d}: {adv:+.4f}  {bar}')

print()
fast_adv = h23_df[h23_df['proc']=='fast'].groupby('n_train')['advantage'].mean()
slow_adv = h23_df[h23_df['proc']=='slow'].groupby('n_train')['advantage'].mean()
if slow_adv.mean() > fast_adv.mean():
    print('  → H3 SUPPORTED: advantage larger on slow-decay process')
else:
    print('  → H3 NOT SUPPORTED: no clear process-type interaction')

if slow_adv.iloc[0] > slow_adv.iloc[-1]:
    print('  → H2 SUPPORTED: advantage larger at small N')
else:
    print('  → H2 NOT SUPPORTED: no clear N interaction')